# 02 — Judge Analysis

Block-normalized judge score distributions and outlier checks. Run `uv run python scripts/derive.py --rebuild` before this notebook so `v_judge_block_stats` is current.


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../scores.db')

raw = pd.read_sql(
    """
    SELECT *
    FROM v_judge_block_stats
    WHERE block_score_count >= 3
    """,
    conn,
)
raw['performance_date'] = pd.to_datetime(raw['performance_date'])
raw.head()


In [ ]:
# Judge workload by normalized score block
judge_workload = (
    raw.groupby('judge')
    .agg(
        score_count=('score_id', 'count'),
        block_count=('score_block_id', 'nunique'),
        first_date=('performance_date', 'min'),
        last_date=('performance_date', 'max'),
    )
    .sort_values(['block_count', 'score_count'], ascending=False)
)
judge_workload.head(20)


In [ ]:
# Block-level spread by class/caption/subcaption
block_spread = pd.read_sql(
    """
    SELECT
        season_year, class_code, caption, subcaption,
        count(*) AS block_count,
        avg(score_count) AS avg_entries,
        avg(score_range) AS avg_range,
        avg(stddev_score) AS avg_stddev
    FROM v_score_blocks
    WHERE score_count >= 3
    GROUP BY season_year, class_code, caption, subcaption
    ORDER BY season_year, class_code, caption, subcaption
    """,
    conn,
)
block_spread.head(30)


In [ ]:
# Largest absolute in-block z-score outliers
outliers = raw[raw['block_z_score'].notna()].copy()
outliers['abs_z'] = outliers['block_z_score'].abs()
outliers = outliers.sort_values('abs_z', ascending=False)
outliers[[
    'performance_date', 'competition_name', 'class_code', 'canonical_ensemble_name',
    'judge', 'caption', 'subcaption', 'score', 'rank', 'block_z_score',
    'block_score_count', 'block_score_range'
]].head(30)


In [ ]:
# Normalized score distribution for the busiest judges
top_judges = judge_workload.head(12).index
plot_df = raw[raw['judge'].isin(top_judges) & raw['block_z_score'].notna()]

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=plot_df, x='judge', y='block_z_score', ax=ax)
ax.axhline(0, color='black', linewidth=1, alpha=0.4)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('Judge score distribution after block normalization')
plt.tight_layout()


In [ ]:
# Candidate judge/ensemble favoritism checks: repeated positive or negative z-scores
favoritism_candidates = (
    raw[raw['block_z_score'].notna()]
    .groupby(['judge', 'canonical_ensemble_id', 'canonical_ensemble_name', 'caption', 'subcaption'])
    .agg(
        observations=('score_id', 'count'),
        mean_z=('block_z_score', 'mean'),
        median_z=('block_z_score', 'median'),
        max_abs_z=('block_z_score', lambda s: s.abs().max()),
    )
    .query('observations >= 5')
    .sort_values(['mean_z', 'observations'], ascending=[False, False])
)
favoritism_candidates.head(30)
